# Module 8 — Neural networks and deep learning (clothing images, Keras): commented walkthrough

Alexey's lecture code (lessons 8.1 – 8.12), with comments that explain **three things** for every step:

- `# GOAL`   what this step is for, in plain words
- `# WHY`    the machine-learning logic behind it (what could go wrong if we skipped it)
- `# PYTHON` how the Python / NumPy / TensorFlow-Keras syntax works, so you can write it yourself later

The original, uncommented notebook is untouched in `../notebooks/`. The images are read from
`../notebooks/clothing-dataset-small/` (one copy for both folders).

**Runtime.** Every `model.fit` here trains a frozen Xception plus a small head. The lecture's settings
(10–50 epochs per model, ~15 models) take hours on a laptop CPU. `QUICK_RUN = True` (first code cell) caps every
run at 3 epochs, enough to see the curves move. Even then the sweeps in 8.6, 8.8 and 8.9 (3–4 models each) and the
299×299 run in 8.11 are the long cells: on the 2-core test machine a 150×150 epoch took ~4 s per batch of 32
(~7 min per epoch); an Apple-Silicon Mac is several times faster. Start a sweep, then read the lesson while it runs.
For the real numbers set `QUICK_RUN = False` and run on a free Kaggle or Colab T4 GPU (see `../lessons/15-install.md`).
Checkpoints (8.7, 8.11) are written next to this notebook and weigh ~85 MB each: don't commit them.

**What was verified before handing this over:** executed top to bottom on Linux, CPU only, TensorFlow 2.21 /
Keras 3.15, with `QUICK_RUN = True` and every `fit`/`evaluate` additionally limited to 2 batches (a test harness,
not part of this notebook). So every cell runs on Keras 3; the accuracy numbers you get are your own.

Differences from the lecture:
- `%autosave 0` and `!git clone git@github.com:...` are dropped: the dataset is already in `../notebooks/`;
  8.1 counts the images per split and class instead;
- one `DATA_DIR` path instead of `'./clothing-dataset-small/...'` typed in every cell;
- `QUICK_RUN` / `n_epochs()` (see above). With `QUICK_RUN = False` the epoch counts are exactly the lecture's;
- **Keras 3 (TensorFlow 2.16+) fixes.** `model.save_weights('model_v1.h5', save_format='h5')` fails twice on Keras 3
  (`save_format` no longer exists, and weight files must end in `.weights.h5`) → `model.save_weights('model_v1.weights.h5')`.
  Checkpoints are saved as `.keras` (the native format) instead of the legacy `.h5`;
- 8.7: the lecture's variable `chechpoint` is spelled `checkpoint`;
- 8.11: the lecture builds the **validation** set with `train_gen`, so validation images get augmented
  (Alexey points out this bug in lessons 8.10 and 8.11). Here it uses `val_gen`;
- 8.12: the checkpoint to load (`xception_v4_1_13_0.903.h5`, hand-typed) is found in code: the file with the
  best validation accuracy in its name. The class names come from `test_ds.class_indices` instead of a typed list;
- 8.12 starts with fresh imports in the lecture (Alexey restarted the kernel). They are kept, harmless;
- the hand-typed axis settings (`plt.xticks(np.arange(10))`, `plt.yticks([0.78, ...])`, `plt.ylim(0.78, 0.86)`) are
  dropped or derived from the history length: with `QUICK_RUN` your curves are shorter and lower;
- `TF_CPP_MIN_LOG_LEVEL=2` hides TensorFlow's start-up log spam (tip from `../lessons/15-install.md`);
- `ImageDataGenerator` is kept because the lecture uses it. Keras 3 marks it as legacy; new code would use
  `keras.utils.image_dataset_from_directory`. The homework is PyTorch anyway: see the dictionary at the end.

## Setup

In [ ]:
# GOAL: imports, data location and the run-length switch for the whole notebook.
import os
# WHY: must be set BEFORE tensorflow is imported, otherwise the C++ log lines are already printed.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# PYTHON: Path objects join with '/', so DATA_DIR / 'train' is the train folder on any OS.
DATA_DIR = Path('../notebooks/clothing-dataset-small')

# True = every training run is capped at 3 epochs (minutes instead of hours on a laptop CPU).
# False = the lecture's epoch counts. Use a GPU (Kaggle / Colab) for that.
QUICK_RUN = True


def n_epochs(full):
    # PYTHON: a conditional expression: value_if_true if condition else value_if_false.
    return min(full, 3) if QUICK_RUN else full

## 8.1 Fashion classification

Task: an online shop wants to suggest the category (t-shirt, pants, shoes, ...) when a user uploads a photo.
That is **multi-class image classification**: 10 classes. The data is the small version of Alexey's clothing
dataset, already split into `train/`, `validation/` and `test/`, one sub-folder per class.

More background: https://cs231n.github.io/ (Stanford's CNN course notes).

In [ ]:
# GOAL: see what we have: how many images per split and per class.
# WHY: the folder name IS the label. A class with very few images will be learnt badly,
#      and the split sizes tell us how noisy the validation accuracy will be (341 images -> 1 image = 0.3%).
# PYTHON: .iterdir() lists a folder; sorted() makes the order stable; a dict comprehension builds {class: count}.
for split in ['train', 'validation', 'test']:
    counts = {d.name: len(list(d.iterdir())) for d in sorted((DATA_DIR / split).iterdir()) if d.is_dir()}
    print(split, sum(counts.values()), counts)

## 8.2 TensorFlow and Keras

**TensorFlow** is the engine (tensors, automatic gradients, CPU/GPU kernels); **Keras** is the high-level API on top
(layers, models, `fit`). Install: `pip install tensorflow` (see START_HERE for the Mac). An image is just a NumPy array:
height × width × 3 colour channels, values 0–255.

In [ ]:
# PYTHON: 'from tensorflow import keras' gives the Keras that ships with this TensorFlow (Keras 3 for TF >= 2.16).
import tensorflow as tf
from tensorflow import keras

tf.__version__, keras.__version__

In [ ]:
# GOAL: load one image from disk.
# PYTHON: load_img returns a PIL image; Jupyter displays it when it is the last expression of the cell.
from tensorflow.keras.preprocessing.image import load_img

path = DATA_DIR / 'train' / 't-shirt'
name = '5f0a3fa0-6a3d-4b68-b213-72766a643de7.jpg'
fullname = path / name
load_img(fullname)

In [ ]:
# GOAL: resize to the size the network expects.
# WHY: a network has a FIXED input shape; Xception was trained on 299 x 299 images.
img = load_img(fullname, target_size=(299, 299))

# PYTHON: np.array(img) turns the picture into an array of shape (height, width, channels), dtype uint8 (0..255).
x = np.array(img)
x.shape

## 8.3 Pre-trained convolutional neural networks

**ImageNet** (https://www.image-net.org/): ~1.2 M photos in 1,000 classes. Keras ships networks already trained on it
(https://keras.io/api/applications/). We take **Xception** and ask it what is on our t-shirt photo.
The first call downloads the weights (~90 MB) into `~/.keras/models/`.

In [ ]:
# PYTHON: every Keras application comes with its OWN preprocess_input and decode_predictions. Import the matching ones.
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.applications.xception import decode_predictions

In [ ]:
# GOAL: the full Xception network, with its 1000-class ImageNet head.
model = Xception(weights='imagenet', input_shape=(299, 299, 3))

In [ ]:
# WHY: models always take a BATCH of images. One image -> a batch of size 1: shape (1, 299, 299, 3).
# PYTHON: np.array([x]) wraps the single array in a list, which adds the leading batch dimension.
X = np.array([x])
X.shape

In [ ]:
# WHY: the network was trained on pixels scaled the Xception way (0..255 -> -1..1). Feed it raw 0..255 values
#      and the predictions are garbage, without any error message. Same preprocessing at training and serving time.
X = preprocess_input(X)

In [ ]:
# GOAL: predict. The output has shape (1, 1000): one probability per ImageNet class.
pred = model.predict(X)
pred.shape

In [ ]:
# GOAL: translate the 1000 numbers into the top-5 class names.
# WHY: ImageNet has 'jersey' and 'sweatshirt' but no 't-shirt', and no class for most of our 10 categories.
#      The network sees the image well; its HEAD answers the wrong question. That is what 8.5 fixes.
decode_predictions(pred)

## 8.4 Convolutional neural networks

No new code in this lesson; read `../lessons/04-conv-neural-nets.md`. The two layer types:

- **Convolutional layer**: a set of small filters (3×3, 5×5) slid over the image. Each filter gives a **feature map**:
  how strongly that local pattern appears at each position. This is a discrete 2-D convolution, the same operation as a
  matched filter in signal processing; the difference is that the filter coefficients are *learnt*. Stacking layers
  turns simple patterns (edges) into complex ones (sleeves, collars). At the end, the image has become a **vector**.
- **Dense layer**: every output is a weighted sum of all inputs (then an activation). Several logistic regressions in
  parallel. With 10 outputs + softmax it turns the vector into 10 class probabilities.

Other layer types (pooling, normalisation, ...): https://cs231n.github.io/

## 8.5 Transfer learning

Keep Xception's convolutional part (it already knows how to see), throw away its ImageNet head, and train only a new
small head for our 10 classes. Small images (150×150) first: ~4× cheaper than 299×299, good enough to compare ideas.

In [ ]:
# GOAL: a generator that reads images from folders, resizes them and applies preprocess_input, batch by batch.
# WHY: 3,000+ images don't need to be in memory at once; the generator streams them from disk during training.
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

# PYTHON: flow_from_directory takes the sub-folder names as class labels and returns an iterator of (X, y) batches.
#         The labels are one-hot vectors (class_mode='categorical' is the default).
train_ds = train_gen.flow_from_directory(
    DATA_DIR / 'train',
    target_size=(150, 150),
    batch_size=32
)

In [ ]:
# The folder -> index mapping (alphabetical). Column i of the model's output means class i of this dict.
train_ds.class_indices

In [ ]:
# PYTHON: next(iterator) pulls one batch: X has shape (32, 150, 150, 3), y has shape (32, 10).
X, y = next(train_ds)

In [ ]:
y[:5]   # one-hot: a single 1 per row, in the column of that image's class

In [ ]:
val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

# WHY: shuffle=False for validation: the order doesn't matter for the score, and a fixed order
#      makes predictions line up with val_ds.filenames / val_ds.classes if you want to inspect them.
val_ds = val_gen.flow_from_directory(
    DATA_DIR / 'validation',
    target_size=(150, 150),
    batch_size=32,
    shuffle=False
)

In [ ]:
# GOAL: the transfer-learning model: frozen Xception body + new 10-class head.
# WHY: include_top=False drops the 1000-class ImageNet head. trainable=False freezes ~20 M weights, so training
#      only adjusts the new Dense layer: fast, and with 3,000 images we could not train Xception from scratch anyway.
base_model = Xception(
    weights='imagenet',
    include_top=False,
    input_shape=(150, 150, 3)
)

base_model.trainable = False

# PYTHON: the Keras FUNCTIONAL API. Each layer is called on the previous output like a function;
#         keras.Model(inputs, outputs) then wraps the whole chain into one model.
inputs = keras.Input(shape=(150, 150, 3))

# WHY: training=False keeps Xception's BatchNormalization layers in inference mode (their running statistics stay
#      frozen), which is what you want for a frozen base.
base = base_model(inputs, training=False)

# WHY: the body outputs a 5 x 5 x 2048 block of feature maps. Averaging each map over the 5 x 5 positions
#      gives ONE vector of 2048 numbers per image: the "vector representation" of 8.4.
vectors = keras.layers.GlobalAveragePooling2D()(base)

# WHY: 10 outputs, one per class, NO activation: these are raw scores (logits). The loss applies softmax itself.
outputs = keras.layers.Dense(10)(vectors)

model = keras.Model(inputs, outputs)

In [ ]:
# GOAL: choose how to learn (optimizer) and what to minimise (loss).
# WHY: Adam = gradient descent with adaptive per-weight step sizes; learning_rate is the main knob (8.6).
learning_rate = 0.01
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

# WHY: categorical cross-entropy = log-loss for several classes. from_logits=True because the last layer
#      has no softmax; letting the loss do softmax + log together is numerically more stable.
#      (HW8's nn.BCEWithLogitsLoss is the same idea for ONE output and two classes.)
loss = keras.losses.CategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

In [ ]:
# GOAL: train. One EPOCH = one pass over all training images; after each epoch Keras scores val_ds.
# PYTHON: fit returns a History object; history.history is a dict of lists: 'loss', 'accuracy', 'val_loss', 'val_accuracy'.
history = model.fit(train_ds, epochs=n_epochs(10), validation_data=val_ds)

In [ ]:
# WHY: plot validation accuracy per epoch. The lecture also had the train curve (commented out): plot both
#      to see the gap; a train curve that keeps rising while validation stalls = overfitting.
#plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.xticks(np.arange(len(history.history['val_accuracy'])))
plt.legend()

## 8.6 Adjusting the learning rate

The learning rate is the step size of gradient descent. Too small: learns slowly. Too large: jumps around and
overfits (Alexey's analogy: reading a book too fast you skim, too slow you never finish). Wrap model creation in a
function so every experiment starts from a fresh model.

In [ ]:
# GOAL: one function = one fresh, compiled model. Every call builds new weights.
# WHY: re-using a trained model for the next experiment would mix the experiments up.
def make_model(learning_rate=0.01):
    base_model = Xception(
        weights='imagenet',
        include_top=False,
        input_shape=(150, 150, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(150, 150, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)
    outputs = keras.layers.Dense(10)(vectors)
    model = keras.Model(inputs, outputs)

    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy(from_logits=True)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
# GOAL: train one model per learning rate and keep each history.
# PYTHON: scores is a dict {learning_rate: history dict}, filled in the loop.
scores = {}

for lr in [0.0001, 0.001, 0.01, 0.1]:
    print(lr)

    model = make_model(learning_rate=lr)
    history = model.fit(train_ds, epochs=n_epochs(10), validation_data=val_ds)
    scores[lr] = history.history

    print()
    print()

In [ ]:
# WHY: drop the two extremes (too slow / too unstable) so the plot compares the two serious candidates.
# PYTHON: del d[key] removes one entry from a dict.
del scores[0.1]
del scores[0.0001]

In [ ]:
for lr, hist in scores.items():
    #plt.plot(hist['accuracy'], label=('train=%s' % lr))
    # PYTHON: 'val=%s' % lr inserts lr into the string (old-style formatting).
    plt.plot(hist['val_accuracy'], label=('val=%s' % lr))

plt.xticks(np.arange(len(hist['val_accuracy'])))
plt.legend()

In [ ]:
# The lecture's choice: 0.001 (similar accuracy to 0.01, smaller train/validation gap).
learning_rate = 0.001

## 8.7 Checkpointing

Validation accuracy goes up and down between epochs. A **checkpoint** saves the model to disk whenever it improves,
so we keep the best epoch, not the last one. Keras does it with a **callback**: code that `fit` calls after each epoch.

In [ ]:
# GOAL: save the current model's weights by hand.
# WHY: Keras 3 rule: weight-only files must end in '.weights.h5' (the lecture's 'model_v1.h5' with
#      save_format='h5' raises an error on Keras 3).
model.save_weights('model_v1.weights.h5')

In [ ]:
# GOAL: a callback that saves the whole model each time val_accuracy reaches a new maximum.
# PYTHON: the {epoch:02d} and {val_accuracy:.3f} placeholders are filled in by Keras -> 'xception_v1_03_0.836.keras'.
# WHY: monitor='val_accuracy' with mode='max' (higher is better); save_best_only=True skips epochs that did not improve.
#      '.keras' is Keras 3's native format; the lecture's '.h5' still works but is marked legacy.
checkpoint = keras.callbacks.ModelCheckpoint(
    'xception_v1_{epoch:02d}_{val_accuracy:.3f}.keras',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [ ]:
learning_rate = 0.001

model = make_model(learning_rate=learning_rate)

# PYTHON: callbacks takes a LIST: you can attach several (early stopping, logging, ...).
history = model.fit(
    train_ds,
    epochs=n_epochs(10),
    validation_data=val_ds,
    callbacks=[checkpoint]
)

## 8.8 Adding more layers

Put one extra dense layer (with ReLU) between the 2048-vector and the output, and try a few sizes.

In [ ]:
# GOAL: same model factory, plus an inner Dense layer of size_inner neurons.
# WHY: ReLU = max(0, x). Without a non-linearity, two stacked Dense layers collapse into one linear map
#      and the extra layer adds nothing.
def make_model(learning_rate=0.01, size_inner=100):
    base_model = Xception(
        weights='imagenet',
        include_top=False,
        input_shape=(150, 150, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(150, 150, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)

    inner = keras.layers.Dense(size_inner, activation='relu')(vectors)

    outputs = keras.layers.Dense(10)(inner)

    model = keras.Model(inputs, outputs)

    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy(from_logits=True)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
learning_rate = 0.001

scores = {}

for size in [10, 100, 1000]:
    print(size)

    model = make_model(learning_rate=learning_rate, size_inner=size)
    history = model.fit(train_ds, epochs=n_epochs(10), validation_data=val_ds)
    scores[size] = history.history

    print()
    print()

In [ ]:
# The lecture fixed the y ticks to 0.78..0.83 (its own numbers); here matplotlib picks them.
for size, hist in scores.items():
    plt.plot(hist['val_accuracy'], label=('val=%s' % size))

plt.xticks(np.arange(len(hist['val_accuracy'])))
plt.legend()

## 8.9 Regularization and dropout

**Dropout**: during training, each epoch randomly switches off a fraction (`droprate`) of the inner layer's
neurons, so the head cannot rely on a few specific features (e.g. a logo on one t-shirt). At prediction time
nothing is dropped. Freezing the base (`trainable = False`) is already a strong form of regularisation.

In [ ]:
# GOAL: the model factory with a Dropout layer after the inner layer.
def make_model(learning_rate=0.01, size_inner=100, droprate=0.5):
    base_model = Xception(
        weights='imagenet',
        include_top=False,
        input_shape=(150, 150, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(150, 150, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)

    inner = keras.layers.Dense(size_inner, activation='relu')(vectors)
    # WHY: Dropout is active only while fitting; model.predict() / evaluate() use all neurons.
    drop = keras.layers.Dropout(droprate)(inner)

    outputs = keras.layers.Dense(10)(drop)

    model = keras.Model(inputs, outputs)

    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy(from_logits=True)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
# WHY: 30 epochs in the lecture: with dropout the model learns more slowly, so it needs more passes.
learning_rate = 0.001
size = 100

scores = {}

for droprate in [0.0, 0.2, 0.5, 0.8]:
    print(droprate)

    model = make_model(
        learning_rate=learning_rate,
        size_inner=size,
        droprate=droprate
    )

    history = model.fit(train_ds, epochs=n_epochs(30), validation_data=val_ds)
    scores[droprate] = history.history

    print()
    print()

In [ ]:
# The lecture zoomed in with plt.ylim(0.78, 0.86); with QUICK_RUN your curves may sit below that window.
for droprate, hist in scores.items():
    plt.plot(hist['val_accuracy'], label=('val=%s' % droprate))

plt.legend()

In [ ]:
# GOAL: compare only no-dropout vs 0.2 (the lecture's pick).
hist = scores[0.0]
plt.plot(hist['val_accuracy'], label=0.0)

hist = scores[0.2]
plt.plot(hist['val_accuracy'], label=0.2)

plt.legend()
#plt.plot(hist['accuracy'], label=('val=%s' % droprate))

## 8.10 Data augmentation

Create new training images from existing ones: flips, rotations, shifts, zoom, shear. A pair of pants flipped
left-right is still a pair of pants. Rules: augment **only the training set**, never validation or test, and keep
only augmentations that make sense for the data (vertical flip: shoes upside down, probably not).

In [ ]:
# WHY: in the lecture's final cell every augmentation is switched off again (vertical_flip is commented out),
#      so this run is the 8.9 model trained longer. To actually try augmentation, add e.g.
#      shear_range=10, zoom_range=0.1, horizontal_flip=True (the set 8.11 keeps) to train_gen ONLY.
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
#     vertical_flip=True,
)

train_ds = train_gen.flow_from_directory(
    DATA_DIR / 'train',
    target_size=(150, 150),
    batch_size=32
)

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_ds = val_gen.flow_from_directory(
    DATA_DIR / 'validation',
    target_size=(150, 150),
    batch_size=32,
    shuffle=False
)

In [ ]:
# WHY: augmented data is harder to memorise, so it needs more epochs (50 in the lecture).
learning_rate = 0.001
size = 100
droprate = 0.2

model = make_model(
    learning_rate=learning_rate,
    size_inner=size,
    droprate=droprate
)

history = model.fit(train_ds, epochs=n_epochs(50), validation_data=val_ds)

In [ ]:
# WHY: train and validation curves together: the gap between them is the overfitting you are fighting.
hist = history.history
plt.plot(hist['val_accuracy'], label='val')
plt.plot(hist['accuracy'], label='train')

plt.legend()

## 8.11 Training a larger model

Same recipe, full 299×299 resolution, the best settings found so far, augmentation on. ~4× the compute per epoch
of the 150×150 runs. This is the run to do on a GPU.

In [ ]:
# GOAL: the model factory with the input size as a parameter.
def make_model(input_size=150, learning_rate=0.01, size_inner=100,
               droprate=0.5):

    base_model = Xception(
        weights='imagenet',
        include_top=False,
        input_shape=(input_size, input_size, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(input_size, input_size, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)

    inner = keras.layers.Dense(size_inner, activation='relu')(vectors)
    drop = keras.layers.Dropout(droprate)(inner)

    outputs = keras.layers.Dense(10)(drop)

    model = keras.Model(inputs, outputs)

    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy(from_logits=True)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
input_size = 299

In [ ]:
# GOAL: augmented training stream, clean validation stream.
# PYTHON: shear_range in degrees, zoom_range=0.1 -> random zoom between 90% and 110%, horizontal_flip -> random mirror.
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    shear_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

train_ds = train_gen.flow_from_directory(
    DATA_DIR / 'train',
    target_size=(input_size, input_size),
    batch_size=32
)


val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

# FIX: the lecture calls train_gen.flow_from_directory here, which AUGMENTS the validation images
#      (random scores between epochs, and it hides the benefit of augmentation). val_gen is the right one.
val_ds = val_gen.flow_from_directory(
    DATA_DIR / 'validation',
    target_size=(input_size, input_size),
    batch_size=32,
    shuffle=False
)

In [ ]:
checkpoint = keras.callbacks.ModelCheckpoint(
    'xception_v4_1_{epoch:02d}_{val_accuracy:.3f}.keras',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [ ]:
learning_rate = 0.0005
size = 100
droprate = 0.2

model = make_model(
    input_size=input_size,
    learning_rate=learning_rate,
    size_inner=size,
    droprate=droprate
)

history = model.fit(train_ds, epochs=n_epochs(50), validation_data=val_ds,
                   callbacks=[checkpoint])

## 8.12 Using the model

Load the best checkpoint, score it **once** on the untouched test set, and classify a single image the way a web
service would (Module 9 turns this into a Lambda function).

In [ ]:
# The lecture restarts the kernel here, hence the repeated imports. Harmless if you didn't restart.
import tensorflow as tf
from tensorflow import keras

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import load_img

from tensorflow.keras.applications.xception import preprocess_input

In [ ]:
# WHY: the test set gets exactly the validation treatment: same size, same preprocess_input, no augmentation.
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_ds = test_gen.flow_from_directory(
    DATA_DIR / 'test',
    target_size=(299, 299),
    batch_size=32,
    shuffle=False
)

In [ ]:
# GOAL: pick the best 8.11 checkpoint without typing its name (the lecture's was 'xception_v4_1_13_0.903.h5').
# PYTHON: Path('.').glob(pattern) finds matching files. The val accuracy is the last '_' part of the stem
#         ('xception_v4_1_13_0.903' -> '0.903'); max(..., key=f) returns the file with the largest f(file).
checkpoints = list(Path('.').glob('xception_v4_1_*.keras'))
best_checkpoint = max(checkpoints, key=lambda p: float(p.stem.split('_')[-1]))
print(best_checkpoint)

model = keras.models.load_model(best_checkpoint)

In [ ]:
# WHY: test accuracy close to the best validation accuracy = the choice of epoch did not just fit validation noise.
# PYTHON: evaluate returns [loss, accuracy] (the metrics given to compile).
model.evaluate(test_ds)

In [ ]:
path = DATA_DIR / 'test' / 'pants' / 'c8d21106-bbdb-4e8d-83e4-bf3d14e54c16.jpg'

In [ ]:
img = load_img(path, target_size=(299, 299))

In [ ]:
import numpy as np

In [ ]:
# GOAL: the serving path, by hand: image -> array -> batch of 1 -> preprocess -> predict.
x = np.array(img)
X = np.array([x])
X.shape

In [ ]:
X = preprocess_input(X)

In [ ]:
pred = model.predict(X)

In [ ]:
# GOAL: class names in column order.
# WHY: the lecture types the 10 names by hand. Reading them from class_indices guarantees the same order
#      as the training labels (a typo or a different order would silently mislabel every prediction).
# PYTHON: sorted(d, key=d.get) sorts the keys by their value (0, 1, 2, ...).
classes = sorted(test_ds.class_indices, key=test_ds.class_indices.get)
classes

In [ ]:
# WHY: these are LOGITS (no softmax in the last layer): the largest one is the prediction, but they are not
#      probabilities. tf.nn.softmax(pred[0]) would turn them into probabilities summing to 1.
dict(zip(classes, pred[0]))

## 8.13 Summary

| Step | Tool | Watch out |
|---|---|---|
| Pre-trained network | `Xception(weights='imagenet')` | always the model's own `preprocess_input` |
| Transfer learning | `include_top=False`, `trainable = False`, new `Dense` head | `training=False` keeps BatchNorm frozen |
| Small first | 150×150 to compare ideas, 299×299 for the final model | 299 costs ~4× per epoch |
| Learning rate | `Adam(learning_rate=...)` | too high = unstable, too low = slow |
| Checkpointing | `ModelCheckpoint(..., save_best_only=True, monitor='val_accuracy', mode='max')` | ~85 MB per file; Keras 3: `.keras`, `.weights.h5` |
| Inner layer | `Dense(size, activation='relu')` | no activation = no gain |
| Dropout | `Dropout(droprate)` after the inner layer | needs more epochs |
| Augmentation | `ImageDataGenerator(shear_range=..., zoom_range=..., horizontal_flip=True)` | training set only |
| Using the model | `load_model`, `evaluate(test_ds)`, `predict` | outputs are logits, not probabilities |

## 8.14 Explore more

- More data (other clothing datasets), and other architectures from https://keras.io/api/applications/
- Albumentations: another library for augmentations
- PyTorch instead of Keras: HW8 is PyTorch. The workshop is in `../notebooks/pytorch/README.md`

## Keras → PyTorch, for HW8

| Keras (this notebook) | PyTorch (HW8) |
|---|---|
| `keras.Model(inputs, outputs)` | `class HairModel(nn.Module)` with `__init__` + `forward` |
| `Conv2D`, `Dense`, `Dropout` | `nn.Conv2d`, `nn.Linear`, `nn.Dropout` |
| `ImageDataGenerator(...).flow_from_directory` | `datasets.ImageFolder(dir, transform=...)` + `DataLoader` |
| `preprocess_input` | `transforms.Normalize(mean, std)` after `ToTensor()` |
| `CategoricalCrossentropy(from_logits=True)` | `nn.CrossEntropyLoss()` (10 classes) / `nn.BCEWithLogitsLoss()` (1 logit) |
| `model.compile(optimizer=Adam(...))` | `optimizer = torch.optim.Adam(model.parameters(), lr=...)` |
| `model.fit(train_ds, epochs=..., validation_data=val_ds)` | your own loop: `zero_grad` → forward → `loss.backward()` → `step` |
| `history.history['val_accuracy']` | the lists you append to yourself each epoch |
| `training=False`, no dropout at predict time | `model.eval()` + `torch.no_grad()` |